# notebook d'expérimentation ....
## objectifs
- analyser VOS données 
- charger et visualiser des spectres 2D ou 1D
- analyser les raies, ajuster des gaussiennes...

$\rightarrow$ **en cours :**
- TD L3 Rennes : version Yveline (avec scipy) + version Pascal (avec specutils + calcul du V/R)


# version L3 Rennes

In [4]:

import numpy as np
import scipy.optimize as opt
from scipy.integrate import trapezoid
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# -------------------------------------------------
# L3 Physique - Univ. Rennes - Winter 2026
# Author : Yveline Lebreton

# -------------------------------------------------
# Constants
plt.rcParams['figure.figsize'] = (8, 5)
ln2 = np.log(2.0)

# --------------------------------------------------
# Gaussian fitting

def gaussian(x, a0, a1, a2):
    """Gaussian profile (FWHM parameterization)."""
    return a0 * np.exp(-ln2 * ((x - a1) / a2) ** 2)

def split_gaussian(x, a0, a1, a2, a3):
    """Gaussian with different left/right widths."""
    return np.where(
        x <= a1,
        gaussian(x, a0, a1, a2),
        gaussian(x, a0, a1, a3)
    )

# --------------------------------------------------
# Coefficient of determination R²

def rsq(y, f):
    stot = np.sum((y - np.mean(y)) ** 2)
    sres = np.sum((y - f) ** 2)
    return 1.0 - sres / stot

# --------------------------------------------------
# Peak detection

def detect_peak_auto_width(wavelength, intensity, approx_lower, approx_upper, fraction=0.5):
    mask = (wavelength >= approx_lower) & (wavelength <= approx_upper)
    x_window = wavelength[mask]
    y_window = intensity[mask]

    if len(x_window) == 0:
        print("Error: No data in the specified window. Exit.")
        sys.exit(1)

    peak_idx = np.argmax(y_window)
    peak_wavelength = x_window[peak_idx]
    peak_intensity = y_window[peak_idx]

    # Find points above fraction*peak for FWHM estimate
    significant_idx = np.where(y_window >= fraction * peak_intensity)[0]
    if len(significant_idx) < 2:
        # fallback: use quarter-width of the window
        fit_lower = peak_wavelength - (x_window[-1]-x_window[0])/4
        fit_upper = peak_wavelength + (x_window[-1]-x_window[0])/4
    else:
        fit_lower = x_window[significant_idx[0]]
        fit_upper = x_window[significant_idx[-1]]

    fwhm_guess = fit_upper - fit_lower

    return peak_wavelength, fit_lower, fit_upper, fwhm_guess

# --------------------------------------------------
# Fit the selected wavelength range with a given fitting function

def fit_profile(fitfun, wavelength, intensity,
                lower_limit, upper_limit,
                initial_guess):

    mask = (wavelength >= lower_limit) & (wavelength <= upper_limit)
    x = wavelength[mask]
    y = intensity[mask]

    if len(x) < 5:
        raise ValueError("Not enough data points in selected range.")

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    popt, pcov = opt.curve_fit(fitfun, x, y, p0=initial_guess)
    perr = np.sqrt(np.diag(pcov))
    yfit = fitfun(x, *popt)

    print("Fit parameters:", *popt)
    print("Parameter uncertainties:", *perr)
    print(f"R²: {rsq(y, yfit):.6f}")

    auc_value = trapezoid(yfit, x)
    print(f"AUC (scipy.trapezoid): {auc_value:.6f}")
    print()

    # Plot
    fig, ax = plt.subplots()
    ax.plot(x, y, label="Data")
    ax.plot(x, yfit, 'r--', label="Fit")
    ax.minorticks_on()
    ax.set_xlabel(r'$\lambda\ (\AA)$', fontsize=14)
    ax.set_ylabel(r'I (ADU)', fontsize=14)
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    return popt, pcov

# --------------------------------------------------
# Main programme

if __name__ == '__main__':

    # Ensure file path
    #base_dir = Path(__file__).resolve().parent
    # corrections plouis : changement des chemins / fichiers
    base_dir = Path('data/TP_Rennes')
    
    # Choose spectrum
    spectrum = ['Spectres/2026/_alcyone_20250930_92.dat',
               'Spectres/2026/_alp_cep_20250930_856.dat',
               'Spectres/2019/ngc6543_20191021_784.dat',
               'Spectres/2019/ngc6543_20191021_784_continuum_off.dat',
               'Spectres/2019/agdra_20191021_877.dat',
               'Spectres/2019/hd145454_20180924_918.dat',
               'Spectres/2026/_pleione_20250930_939.dat',
                ]

    print("Available spectra : alcyone (0), alp cep (1), ngc6543 (2), ngc6543 continuum (3), agdra (4), hd145454 (5), pleione (6)")
    i = int(input("Choose your spectrum: "))
    #i=0
    file_path = base_dir/spectrum[i]

    if not file_path.exists():
        print(f"Error: File not found: {file_path}")
        sys.exit(1)

    data = np.genfromtxt(file_path)
    wavelength = data[:, 0]
    intensity = data[:, 1]

    # Full spectrum plot
    fig, ax = plt.subplots()
    ax.plot(wavelength, intensity)
    ax.minorticks_on()
    ax.set_xlabel(r'$\lambda\ (\AA)$', fontsize=14)
    ax.set_ylabel(r'I (ADU)', fontsize=14)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    proceed = input("Do you want to perform the Gaussian fit? (y/n): ").strip().lower()
    if proceed not in ('y', 'yes', 'o', 'oui'):
      print("Fitting skipped by user. Exiting program.")
      sys.exit(0)  

    # Approximate expected line window
    print("Fitting of a specified line:")
    approx_lower = float(input("Enter approximate lower wavelength: "))
    approx_upper = float(input("Enter approximate upper wavelength: "))

    # Automatic peak detection and FWHM
    peak, lower_limit, upper_limit, a2_guess = detect_peak_auto_width(
        wavelength, intensity, approx_lower, approx_upper
    )
    print(f"Detected peak at λ = {peak:.2f} Å")
    print(f"Fitting window: {lower_limit:.2f} Å → {upper_limit:.2f} Å")
    print(f"Initial FWHM guess: {a2_guess:.2f} Å\n")

    # Initial amplitude guess
    idx = (wavelength >= lower_limit) & (wavelength <= upper_limit)
    a0_guess = intensity[idx].max()

    # Gaussian fit
    print("Gaussian fit:")
    initial_guess_gauss = (a0_guess, peak, a2_guess)
    fit_profile(gaussian, wavelength, intensity, lower_limit, upper_limit, initial_guess_gauss)

    # Split-Gaussian fit
    print("Split-Gaussian fit:")
    initial_guess_split = (a0_guess, peak, a2_guess, a2_guess)
    fit_profile(split_gaussian, wavelength, intensity, lower_limit, upper_limit, initial_guess_split)

Available spectra : alcyone (0), alp cep (1), ngc6543 (2), ngc6543 continuum (3), agdra (4), hd145454 (5), pleione (6)


Choose your spectrum:  9


IndexError: list index out of range

# version SAR 

In [1]:
%matplotlib widget
import numpy as np
from spectro_dashboard import SpectroDashboard

# 1. Afficher le dashboard
db = SpectroDashboard()
db.show()


In [5]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
from specutils import Spectrum, SpectralRegion
from astropy import units as u
from specutils.analysis import centroid, fwhm, equivalent_width, snr, snr_derived


In [6]:
### récupération de la vitesse barycentrique de pleione

import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.constants import c

c_kms = c.to('km/s')      #c = 299792.458 km/s

### coord du CALC
obs_longitude = 1.5 * u.Unit('deg')
obs_latitude = 48.0 * u.Unit('deg')
obs_height = 50 * u.Unit('m')

obs_loc = EarthLocation(lat = obs_latitude, lon = obs_longitude, height = obs_height)
obs_time = Time('2025-09-30 23:00:00')

# on récupère ses coords (attention : syntaxe SIMBAD)
target_coord = SkyCoord.from_name('pleione')
print(target_coord)

obs_coord = EarthLocation(lon = obs_longitude, lat = obs_latitude)

# vitesse héliocentrique
#heliocorr = target_coord.radial_velocity_correction('heliocentric', obstime=obs_time, location=obs_loc)
heliocorr = target_coord.radial_velocity_correction('barycentric', obstime=obs_time, location=obs_loc)
heliocorr.to(u.km / u.s)


<SkyCoord (ICRS): (ra, dec) in deg
    (57.29673582, 24.1367102)>


<Quantity 23.55185251 km / s>

In [7]:
#base_dir = Path(__file__).resolve().parent
# corrections plouis : changement des chemins / fichiers
base_dir = Path('data/TP_Rennes')

# Choose spectrum
spectrum = ['Spectres/2026/_alcyone_20250930_392.dat',
           'Spectres/2026/_alp_cep_20250930_856.dat',
           'Spectres/2019/ngc6543_20191021_784.dat',
           'Spectres/2019/ngc6543_20191021_784_continuum_off.dat',
           'Spectres/2019/agdra_20191021_877.dat',
           'Spectres/2019/hd145454_20180924_918.dat',
           'Spectres/2026/_pleione_20250930_939.dat',
            ]

#print("Available spectra : alcyone (0), alp cep (1), ngc6543 (2), ngc6543 continuum (3), agdra (4), hd145454 (5), pleione (6)")
#i = int(input("Choose your spectrum: "))
i=6
file_path = base_dir/spectrum[i]

if not file_path.exists():
    raise(ValueError(f"Error: File not found: {file_path}"))
    
# on charge le spectre
_spc_array = np.loadtxt(file_path)
_spec1d = Spectrum(spectral_axis=_spc_array[:,0] * u.Unit('Angstrom'), flux=_spc_array[:,1] * u.Unit('mJy'))

# on décale de la vitesse barycentrique
_spec1d = Spectrum(spectral_axis=_spec1d.spectral_axis * (1 + heliocorr /c), flux=_spec1d.flux)

db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='brut_'+file_path.stem[1:5]+'...', color='green')


db.show_spectrum(_spec1d.wavelength, _spec1d.flux, label='corr_'+file_path.stem[1:5]+'...', color='red')


--> Affichage du spectre brut_plei... : dispersion=0.0370 Å/px
--> Affichage du spectre corr_plei... : dispersion=0.0370 Å/px


In [8]:
# ATTENTION : le spectre autour de H alpha d'une étoile Be est un 'shell', i.e. une enveloppe de gaz autour de l'étoile
# -> le modèle à ajuster n'est PAS une double gaussienne mais une large gaussienne en émission, 'percée' d'une autre gaussienne en absorption

import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation

# on définit une zone autour des cibles
lambda_ha = 6562.82
lambda_HeI = 6678.15 
window_width = 20       # nb angstroms de chaque côté

# on prépare les données fixes pour l'incertitude systématique (instrument = starEx 2400)
fwhm_neon_pix = 5.0   # largeur raie néon en px, retourné par specinti
disp_physique = 0.0622 # dispersion du starEx2400, retourné par specinti
R_power = 19000 
fwhm_inst = lambda_ha / R_power # largeur instrumentale

# on utilise les valeurs brutes (pour éviter les soucis de numpy avec les unités astropy)
x_data = _spec1d.spectral_axis.value
y_data = _spec1d.flux.value 

# on découpe la zone 
mask = (x_data > lambda_ha - window_width) & (x_data < lambda_ha + window_width)
x_win = x_data[mask]
y_win = y_data[mask]

# on prépare les valeurs d'aide pour le fitter
y_min = np.min(y_win)
y_max = np.max(y_win)
y_cont_guess = y_min
amp_em_guess = y_max - y_min
amp_abs_guess = (y_max - y_min) * 0.5 # On suppose que l'absorption mange la moitié du shell

# on initialise les 3 modèles
model_init = (models.Gaussian1D(amplitude=amp_em_guess, mean=lambda_ha, stddev=2.5) +     # Emission large
              models.Gaussian1D(amplitude=amp_abs_guess, mean=lambda_ha, stddev=0.5) +    # Absorption fine
              models.Const1D(amplitude=1.0))                                              # Continuum

# on ajuste
fitter = fitting.LevMarLSQFitter()
fit_result = fitter(model_init, x_win, y_win)

# on extrait les données de la large gausssienne en emission
stddev_mesure = fit_result[0].stddev.value
fwhm_mesure = 2.355 * stddev_mesure      # FWHM = 2.355 * sigma (2.355 = 2 * sqrt(2 * ln 2) pour une gaussienne)

print("-" * 40)
print(f"FWHM Mesurée                     : {fwhm_mesure:.3f} +/- {stddev_mesure:.2f} A")

# on recherche maintenant les deux pics V et R sur la courbe fittée
# on prend les maxima de part et d'autre du centre estimé (halpha)
y_fit = fit_result(x_win)
mask_V = x_win < lambda_ha
mask_R = x_win > lambda_ha

# on trouve les index des maximum dans le masque
idx_V_relatif = np.argmax(y_fit[mask_V])
idx_R_relatif = np.argmax(y_fit[mask_R])

# on récupére la position (en Angströms)
x_V = x_win[mask_V][idx_V_relatif]
x_R = x_win[mask_R][idx_R_relatif]

# on récupère les intensité relatives
i_V = y_fit[mask_V][idx_V_relatif]
i_R = y_fit[mask_R][idx_R_relatif]

# on en déduit le V/R
v_sur_r = (i_V - 1) / (i_R - 1)

# et la vitesse de rotation du disque
delta_lambda = x_R - x_V
v_rot_disk = (c_kms * (delta_lambda / lambda_ha)) / 2

# on établit l'incertitude finale à partir du néon : on prend 1/10 de la FWHM d'une raie du néon
fwhm_inst_a = fwhm_neon_pix * disp_physique  # Résolution en Angströms
err_pos = fwhm_inst_a / 10.0                 

err_delta_lambda = np.sqrt(err_pos**2 + err_pos**2)     # on propage car il y a 2 pics
err_v = c * (err_delta_lambda / (2 * lambda_ha))         
    
print("-" * 40)
print(f"Pic Bleu (V)                     : {x_V:.3f} +/- {err_pos:.3f} A")
print(f"Pic Rouge (R)                    : {x_R:.3f} +/- {err_pos:.3f} A")
print(f"Séparation (dL)                  : {delta_lambda:.3f} +/- {err_delta_lambda:.03f} A")
print(f"Rapport V/R (asymétrie disque)   : {v_sur_r:.2f}")
print(f"Vitesse du gaz au bord disque    : {v_rot_disk.value:.1f} +/- {err_v.to('km/s').value:.1f} km/s")
print("-" * 40)

# on affiche le tout
x_plot = np.linspace(x_win.min(), x_win.max(), 1000)

db.clear_spectra()
db.show_spectrum(x_win, y_win, label='brut')
db.show_spectrum(x_win, y_fit, label='fit')

# on affiche les Lignes V/R
db.ax_spec.axvline(x_V, color='blue', linestyle='-', alpha=0.8, label=f'Centre V ({x_V:.2f} A)')
db.ax_spec.axvline(x_R, color='green', linestyle='-', alpha=0.8, label=f'Centre R ({x_R:.2f} A)')
db.ax_spec.legend()



----------------------------------------
FWHM Mesurée                     : 6.616 +/- 2.81 A
----------------------------------------
Pic Bleu (V)                     : 6561.496 +/- 0.031 A
Pic Rouge (R)                    : 6564.147 +/- 0.031 A
Séparation (dL)                  : 2.650 +/- 0.044 A
Rapport V/R (asymétrie disque)   : 0.94
Vitesse du gaz au bord disque    : 60.5 +/- 1.0 km/s
----------------------------------------
--> Affichage du spectre brut : dispersion=0.0370 Å/px
--> Affichage du spectre fit : dispersion=0.0370 Å/px
